# Hybrid search
In this notebook i implement a hybrid search where i use both the results of keyword search and vector search to retrieve the needed context to pass to the llm

In [1]:
from dotenv import load_dotenv
# initializing the env vars in the parent of the current directory
load_dotenv("../.env")

True

In [2]:
import os
from pgvecto_rs.sdk import PGVectoRs
from llama_index.vector_stores.pgvecto_rs import PGVectoRsStore
from llama_index.core import VectorStoreIndex

DATABASE_URL = (
    "postgresql+psycopg://{username}:{password}@{host}:{port}/{db_name}".format(
        port=os.getenv("DB_PORT", "5432"),
        host=os.getenv("DB_HOST", "localhost"),
        username=os.getenv("DB_USER", "postgres"),
        password=os.getenv("DB_PASS", "mysecretpassword"),
        db_name=os.getenv("DB_NAME", "postgres"),
    )
)

client = PGVectoRs(
    db_url=DATABASE_URL,
    collection_name="items",
    dimension=1536,  # Using OpenAI’s text-embedding-ada-002
)

vector_store = PGVectoRsStore(client=client)
index = VectorStoreIndex.from_vector_store(vector_store)


## Basic Implementation
In this first implementation the nodes are loaded from the db into memory every time and the indexes are recreated, which can be quite slow in production for a large number of nodes

In [ ]:
# let us first fetch all the nodes from the db directly
import json
from sqlalchemy import create_engine, text
from llama_index.core.schema import TextNode, BaseNode

engine = create_engine(url=DATABASE_URL)

conn = engine.connect()

rows = conn.execute(
    text("SELECT id, text, meta, embedding FROM collection_items")
).fetchall()

all_nodes: list[BaseNode] = [
    TextNode(
        text=row.text,
        id_=str(row.id),
        metadata=row.meta or [],
        embedding=json.loads(row.embedding) if isinstance(row.embedding, str) else list(row.embedding)
    ) for row in rows
]

print(f"""
      Fetched {len(all_nodes)} nodes
      
      node1:
      {all_nodes[0] if len(all_nodes) > 0 else "None"}
    """)


      Fetched 38 nodes

      node1:
      Node ID: c28007d4-bd26-4307-8049-85566c92748c
Text: What I Worked On  February 2021  Before college the two main
things I worked on, outside of school, were writing and programming. I
didn't write essays. I wrote what beginning writers were supposed to
write then, and probably still are: short stories. My stories were
awful. They had hardly any plot, just characters with strong feelings,
which I ...
    


In [12]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.retrievers.fusion_retriever import FUSION_MODES

bm2_retriever = BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=5)

vector_retriever = index.as_retriever()

hybrid_retriever = QueryFusionRetriever(
    retrievers=[bm2_retriever, vector_retriever],
    similarity_top_k=5, # the final top-k after fusion from the two
    num_queries=1, # deliberately set to one to avoid query generation
    use_async=True,
    verbose=True,
    mode=FUSION_MODES.RECIPROCAL_RANK
)

In [14]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(retriever=hybrid_retriever)

In [16]:
# testing the query_engine

response = query_engine.query("Describe the writer in less than 10 words")
print(response)

Writer, programmer, entrepreneur, insightful, influential, innovative, experienced, reflective.


## Better Implementation
In this implementation we avoid the need to load all the nodes into memory and simply create our own custorm HyBridFusser that uses both bm25 and the vector search

The downside though is that in this one you would probably need to create your own logic when it comes to rewriting the user query with ai when using it in the keyword search. unline in QueryFussionRetriever where you can simply set the num_queries to be above 1 so that such is done for you.